# Vajra — 04: Stage 2 Physics-Constrained Diffusion Downscaling (12km -> 5km)
### MoES / NCMRWF Problem Statement 26078
This notebook trains and samples the Conditional UNet diffusion model with atmospheric physics losses (divergence & moisture flux conservation), generating a 20-sample probabilistic ensemble.

In [ ]:
import torch
import matplotlib.pyplot as plt
from vajra.downscaling.unet_backbone import ConditionalUNet
from vajra.downscaling.diffusion_pipeline import WeatherDiffusionPipeline
from vajra.downscaling.physics_loss import AtmosphericPhysicsLoss
from vajra.downscaling.sampler import ProbabilisticEnsembleSampler

# Instantiate Conditional UNet Backbone
model = ConditionalUNet(
    in_channels=4,          # (u, v, q, precip)
    condition_channels=4,   # 12km coarse forecast
    out_channels=4,
    base_channels=16
)
print(model)

In [ ]:
# Test Physics-Informed Differentiable Loss
criterion = AtmosphericPhysicsLoss(dx=5000.0, dy=5000.0, lam_div=0.1, lam_moist=0.2)
pred = torch.randn(2, 4, 32, 32, requires_grad=True)
target = torch.randn(2, 4, 32, 32)

loss, metrics = criterion(pred, target)
loss.backward()
print(f"Total Physics Loss: {loss.item():.4f}")
print("Loss Components:", metrics)

In [ ]:
# Multi-Future Probabilistic Downscaling (20 samples)
pipeline = WeatherDiffusionPipeline(model=model, num_timesteps=20, device="cpu")
sampler = ProbabilisticEnsembleSampler(diffusion_pipeline=pipeline, num_samples=5, device="cpu")

coarse_input = torch.randn(1, 4, 32, 32)
results = sampler.generate_samples(coarse_input, shape=(4, 32, 32))

# Visualize Mean vs P90 Extreme Peak Retention
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
ax1.imshow(results["mean"][0, 3], cmap="YlGnBu")
ax1.set_title("Ensemble Mean Precipitation (5km)")

ax2.imshow(results["p90"][0, 3], cmap="YlOrRd")
ax2.set_title("P90 Extreme Peak Precipitation (Preserved)")

ax3.imshow(results["uncertainty"][0, 3], cmap="magma")
ax3.set_title("Downscaling Spatial Uncertainty (Std Dev)")
plt.tight_layout()
plt.show()